# 4. Downsampling / matched-training-size experiment.

Both years use TRAIN_SIZE = 3008 (=80% of 2023's 3760).
One model per (year, seed, model, feature) is trained, then evaluated on BOTH the same-yhear held-out set (within) and the full other year (cross).

If the 2022/2023 asymmetry is a sample-size artifacts, it should vanish here.

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier)
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

OUT_DIR = Path("./nsduh_analysis_outputs")

In [ ]:
# 0. Data & Features: identical to 01_crossyear

df_ml = pd.read_csv(OUT_DIR / "df_corrected_7970_with_gpt_profiles_embeddings.csv")
X_embed = np.vstack(df_ml["predictive_embedding"].apply(json.loads).values).astype("float32")
y = df_ml["cost_barrier"].astype(int).values

raw_cols = ["year", "AGE3", "IRINSUR4", "substance_peer_support", "mental_health_peer_support"]
df_raw = df_ml[raw_cols].copy()

for col in raw_cols:
    df_raw[col] = df_raw[col].astype(str)
X_raw = pd.get_dummies(df_raw, drop_first=False).values.astype("float32")
X_combined = np.hstack([X_raw, X_embed]).astype("float32")


feature_sets = {
    "Raw structured": X_raw,
    "GPT profile embedding": X_embed,
    "Raw + GPT profile embedding": X_combined
}

In [7]:
# 1. Models: identical to 01_crossyear
def make_model(name):
    if name == "Logistic Regression":
        return make_pipeline(StandardScaler(), 
                             LogisticRegression(max_iter=3000, solver="saga", 
                                                class_weight="balanced", n_jobs=-1, random_state=42))
    elif name == "Random Forest":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                      n_jobs=-1, class_weight="balanced", random_state=42)
    elif name == "Extra Trees":
        return ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, 
                                    n_jobs=-1, class_weight="balanced", random_state=42)
    elif name == "XGBoost":
        return XGBClassifier(n_estimators=300, learning_rate=0.05,
                             max_leaf_nodes=31, l2_regularization=0.01, 
                             random_state=42)
    elif name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, 
                                              max_leaf_nodes=31, l2_regularization=0.01, 
                                              random_state=42)
    else:
        raise ValueError(f"Unknown model name: {name}")
    

def get_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


model_names = ["Logistic Regression", "Random Forest", "Extra Trees", 
               "XGBoost", "HistGradientBoosting"]

In [11]:
# 2. Matched-training-size experiment

TRAIN_SIZE = 3008   # 80% of 3760 -> same as 2023's within-year train size
seeds = [0, 1, 2, 3, 4]
year_values = df_ml["year"].astype(int).values
year_idx = {yr:np.where(year_values==yr)[0] for yr in (2022, 2023)}

results = []

for seed in seeds:
    for yr in (2022, 2023):
        other = 2023 if yr == 2022 else 2022
        idx = year_idx[yr]
        tr_idx, held_idx = train_test_split(idx, train_size=TRAIN_SIZE,
                                            random_state=seed, stratify=y[idx])
        cross_idx = year_idx[other]
        for feat_name, X in feature_sets.items():
            for m_name in model_names:
                model = make_model(m_name)
                model.fit(X[tr_idx], y[tr_idx])
                auc_within = roc_auc_score(y[held_idx], get_score(model, X[held_idx]))
                auc_cross = roc_auc_score(y[cross_idx], get_score(model, X[cross_idx]))
                results.append({
                    "Seed": seed,
                    "Train Year": yr,
                    "Test Year (cross)": other,
                    "Model": m_name,
                    "Feature": feat_name,
                    "Train Size": TRAIN_SIZE,
                    "Within AUC": auc_within,
                    "Cross AUC": auc_cross,
                    "Drop": auc_within - auc_cross,
                })
                print(f"[seed {seed}] {yr} -> {other} | {m_name:22s} | {feat_name:28s} | "
                      f"within={auc_within:.4f} cross={auc_cross:.4f} "
                      f"drop={auc_within - auc_cross:+.4f}", flush=True)

res = pd.DataFrame(results)
res.to_csv(OUT_DIR / "downsampled_matched_train_results.csv", index=False)

[seed 0] 2022 -> 2023 | Logistic Regression    | Raw structured               | within=0.6667 cross=0.6666 drop=+0.0001


/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[seed 0] 2022 -> 2023 | Random Forest          | Raw structured               | within=0.6633 cross=0.6630 drop=+0.0004
[seed 0] 2022 -> 2023 | Extra Trees            | Raw structured               | within=0.6633 cross=0.6631 drop=+0.0002


/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:14:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "l2_regularization", "max_leaf_nodes" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[seed 0] 2022 -> 2023 | XGBoost                | Raw structured               | within=0.6652 cross=0.6622 drop=+0.0030
[seed 0] 2022 -> 2023 | HistGradientBoosting   | Raw structured               | within=0.6630 cross=0.6641 drop=-0.0011


...

[seed 4] 2023 -> 2022 | XGBoost                | GPT profile embedding        | within=0.6426 cross=0.6862 drop=-0.0436
[seed 4] 2023 -> 2022 | HistGradientBoosting   | GPT profile embedding        | within=0.6436 cross=0.6780 drop=-0.0344


/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[seed 4] 2023 -> 2022 | Logistic Regression    | Raw                          | within=0.6432 cross=0.5806 drop=+0.0626
[seed 4] 2023 -> 2022 | Random Forest          | Raw                          | within=0.6478 cross=0.6972 drop=-0.0493
[seed 4] 2023 -> 2022 | Extra Trees            | Raw                          | within=0.6501 cross=0.7001 drop=-0.0500


/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [01:04:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "l2_regularization", "max_leaf_nodes" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[seed 4] 2023 -> 2022 | XGBoost                | Raw                          | within=0.6438 cross=0.6854 drop=-0.0416
[seed 4] 2023 -> 2022 | HistGradientBoosting   | Raw                          | within=0.6440 cross=0.6771 drop=-0.0331


In [13]:
# 3 Summaries

# (a) Mean drop by direction / model / feature - is the asymmetry gone?
drop_summary = (res.groupby(["Train Year", "Model", "Feature"])["Drop"].
                agg(["mean", "std"]).round(4).reset_index())

drop_summary.to_csv(OUT_DIR / "downsampled_matched_train_drop_summary.csv", index=False)
print("\n== Mean drop by direction (matched train size) ==")
print(drop_summary.to_string(index=False))

# (b) GPT-minuis-Raw drop difference by model (mean over both directions)
pivot = res.pivot_table(index=["Seed", "Train Year", "Model"],
                        columns="Feature", values="Drop").reset_index()
pivot["GPT_drop_minus_Raw_drop"] = (pivot["GPT profile embedding"] - pivot["Raw structured"])
gpt_vs_raw = (pivot.groupby("Model")["GPT_drop_minus_Raw_drop"].agg(["mean", "std"]).round(4))

gpt_vs_raw.to_csv(OUT_DIR / "downsampled_matched_train_gpt_minus_raw.csv")
print("\n=== GPT minus Raw drop difference by model ===")
print(gpt_vs_raw.to_string())


print("\nDone.")



== Mean drop by direction (matched train size) ==
 Train Year                Model               Feature    mean    std
       2022          Extra Trees GPT profile embedding  0.0087 0.0124
       2022          Extra Trees                  Raw   0.0073 0.0116
       2022          Extra Trees        Raw structured  0.0129 0.0139
       2022 HistGradientBoosting GPT profile embedding  0.0434 0.0218
       2022 HistGradientBoosting                  Raw   0.0405 0.0192
       2022 HistGradientBoosting        Raw structured  0.0116 0.0131
       2022  Logistic Regression GPT profile embedding  0.0358 0.0118
       2022  Logistic Regression                  Raw   0.0353 0.0111
       2022  Logistic Regression        Raw structured  0.0079 0.0081
       2022        Random Forest GPT profile embedding  0.0045 0.0118
       2022        Random Forest                  Raw   0.0041 0.0139
       2022        Random Forest        Raw structured  0.0111 0.0121
       2022              XGBoost GPT pr